<a href="https://colab.research.google.com/github/niwatoro/search_edgar/blob/main/Search_SEC_EDGAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import json
import re
import time
from io import StringIO
from pathlib import Path
from typing import Any, Literal

import pandas as pd
import requests
from tqdm.notebook import tqdm

In [ ]:
USER_AGENT = "yazaki@niwatoro.com"
SLEEP_SEC = 0.15


MatchMode = Literal["exact", "contains", "regex"]

In [ ]:
class EdgarClient:
    def __init__(self, user_agent: str, sleep_sec: float = 0.15) -> None:
        if not user_agent or "@" not in user_agent:
            raise ValueError("SEC向けUser-Agentには組織名・連絡先メールを含めてください。")

        self.sleep_sec = sleep_sec
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": user_agent,
                "Accept-Encoding": "gzip, deflate",
            }
        )

    def get(self, url: str) -> requests.Response:
        last_error: Exception | None = None

        for attempt in range(5):
            try:
                r = self.session.get(url, timeout=60)

                if r.status_code in {429, 500, 502, 503, 504}:
                    time.sleep((2 ** attempt) + self.sleep_sec)
                    continue

                r.raise_for_status()
                time.sleep(self.sleep_sec)
                return r

            except Exception as e:
                last_error = e
                time.sleep((2 ** attempt) + self.sleep_sec)

        raise RuntimeError(f"Request failed after retries: {url}") from last_error

    def get_json(self, url: str) -> dict[str, Any]:
        return self.get(url).json()

    def get_text(self, url: str) -> str:
        return self.get(url).text


def normalize_cik(cik: str | int) -> str:
    digits = re.sub(r"\D", "", str(cik))
    if not digits:
        raise ValueError("CIKに数字が含まれていません。")
    if len(digits) > 10:
        raise ValueError("CIKは最大10桁です。")
    return digits.zfill(10)


def cik_for_archive_path(cik10: str) -> str:
    return str(int(cik10))


def accession_no_dashes(accession_number: str) -> str:
    return accession_number.replace("-", "")


def normalize_text(x: Any) -> str:
    return re.sub(r"\s+", " ", str(x).strip()).casefold()


def month_bounds(year_month: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    """
    year_month: 'YYYY-MM'
    return: [start, end) の半開区間
    """
    start = pd.Timestamp(f"{year_month}-01")
    end = start + pd.DateOffset(months=1)
    return start, end


def recent_filings_to_df(recent: dict[str, list[Any]]) -> pd.DataFrame:
    return pd.DataFrame(recent)


def add_edgar_urls(df: pd.DataFrame, cik10: str) -> pd.DataFrame:
    x = df.copy()

    cik_path = cik_for_archive_path(cik10)
    x["cik"] = cik10
    x["accession_no_dashes"] = x["accessionNumber"].astype(str).map(accession_no_dashes)

    x["accession_dir_url"] = (
        "https://www.sec.gov/Archives/edgar/data/"
        + cik_path
        + "/"
        + x["accession_no_dashes"]
        + "/"
    )

    x["filing_detail_url"] = (
        x["accession_dir_url"]
        + x["accessionNumber"].astype(str)
        + "-index.html"
    )

    x["complete_submission_url"] = (
        x["accession_dir_url"]
        + x["accessionNumber"].astype(str)
        + ".txt"
    )

    x["directory_index_json_url"] = x["accession_dir_url"] + "index.json"

    def primary_doc_url(row: pd.Series) -> str | None:
        doc = row.get("primaryDocument")
        if pd.isna(doc) or str(doc).strip() == "":
            return None
        return str(row["accession_dir_url"]) + str(doc)

    x["primary_doc_url"] = x.apply(primary_doc_url, axis=1)

    return x


def get_all_submission_metadata(
    cik: str | int,
    client: EdgarClient,
) -> tuple[str, dict[str, Any], pd.DataFrame]:
    """
    CIKを直接指定して、Submissions API上の全提出履歴metadataを取得する。
    """
    cik10 = normalize_cik(cik)

    root_url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    root = client.get_json(root_url)

    frames: list[pd.DataFrame] = []

    recent = root.get("filings", {}).get("recent")
    if recent:
        frames.append(recent_filings_to_df(recent))

    for file_info in root.get("filings", {}).get("files", []):
        name = file_info["name"]
        old_url = f"https://data.sec.gov/submissions/{name}"
        old_json = client.get_json(old_url)
        frames.append(recent_filings_to_df(old_json))

    if frames:
        df = pd.concat(frames, ignore_index=True)
    else:
        df = pd.DataFrame()

    if df.empty:
        return cik10, root, df

    required_cols = [
        "accessionNumber",
        "filingDate",
        "reportDate",
        "acceptanceDateTime",
        "act",
        "form",
        "fileNumber",
        "filmNumber",
        "items",
        "size",
        "isXBRL",
        "isInlineXBRL",
        "primaryDocument",
        "primaryDocDescription",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df["filingDate_dt"] = pd.to_datetime(df["filingDate"], errors="coerce")
    df = add_edgar_urls(df, cik10)

    return cik10, root, df


def filter_filings_by_month_and_primary_doc_description(
    filings: pd.DataFrame,
    year_month: str,
    primary_doc_description: str | list[str] | None = None,
    match_mode: MatchMode = "exact",
    form_types: list[str] | None = None,
) -> pd.DataFrame:
    """
    指定月とprimaryDocDescriptionでfiling metadataを絞り込む。

    primary_doc_description:
      None: 絞り込みなし
      str: 単一条件
      list[str]: 複数条件。exact/contains/regexの各modeでOR条件。

    match_mode:
      exact: 空白・大文字小文字を正規化して完全一致
      contains: 大文字小文字を無視して部分一致
      regex: 正規表現
    """
    start, end = month_bounds(year_month)

    x = filings.copy()
    x = x[(x["filingDate_dt"] >= start) & (x["filingDate_dt"] < end)]

    if form_types is not None:
        x = x[x["form"].isin(form_types)]

    if primary_doc_description is not None:
        patterns = (
            [primary_doc_description]
            if isinstance(primary_doc_description, str)
            else list(primary_doc_description)
        )

        desc = x["primaryDocDescription"].fillna("").astype(str)

        if match_mode == "exact":
            normalized_patterns = {normalize_text(p) for p in patterns}
            mask = desc.map(normalize_text).isin(normalized_patterns)

        elif match_mode == "contains":
            escaped = [re.escape(p) for p in patterns]
            combined = "|".join(escaped)
            mask = desc.str.contains(combined, case=False, regex=True, na=False)

        elif match_mode == "regex":
            combined = "|".join(patterns)
            mask = desc.str.contains(combined, case=False, regex=True, na=False)

        else:
            raise ValueError(f"Unsupported match_mode: {match_mode}")

        x = x[mask]

    return x.sort_values(["filingDate_dt", "accessionNumber"]).reset_index(drop=True)


def extract_document_table_from_detail_html(
    html: str,
    accession_dir_url: str,
) -> pd.DataFrame:
    """
    EDGARの accession-number-index.html から
    'Seq / Description / Document / Type / Size' 等の提出文書テーブルを抽出する。
    """
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        return pd.DataFrame()

    candidate_tables: list[pd.DataFrame] = []

    for table in tables:
        table = table.copy()
        table.columns = [str(c).strip() for c in table.columns]

        cols = set(table.columns)
        if {"Document", "Type"}.issubset(cols):
            candidate_tables.append(table)

    if not candidate_tables:
        return pd.DataFrame()

    docs = pd.concat(candidate_tables, ignore_index=True)

    def build_doc_url(doc: Any) -> str | None:
        if pd.isna(doc):
            return None
        doc_s = str(doc).strip()
        if not doc_s or doc_s.lower() == "nan":
            return None
        if doc_s.startswith("http://") or doc_s.startswith("https://"):
            return doc_s
        return accession_dir_url + doc_s

    docs["document_url"] = docs["Document"].map(build_doc_url)

    return docs


def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)[:180]


def download_selected_primary_doc_htmls(
    selected_filings: pd.DataFrame,
    client: EdgarClient,
    out_dir: str | Path,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    selected_filings に含まれる primary_doc_url のHTML本文を保存する。

    入力:
      selected_filings:
        filter_filings_by_month_and_primary_doc_description() 後のDataFrame。
        少なくとも accessionNumber, filingDate, form, primaryDocDescription,
        primaryDocument, primary_doc_url を含む想定。

      client:
        EdgarClient インスタンス。

      out_dir:
        保存先ディレクトリ。

      overwrite:
        Falseの場合、既存ファイルは再取得しない。

    戻り値:
      保存結果のDataFrame。
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    records: list[dict[str, Any]] = []

    for _, row in tqdm(selected_filings.iterrows(), total=len(selected_filings)):
        url = row.get("primary_doc_url")

        if pd.isna(url) or str(url).strip() == "":
            records.append(
                {
                    "accessionNumber": row.get("accessionNumber"),
                    "status": "skipped",
                    "reason": "primary_doc_url is missing",
                    "source_url": None,
                    "saved_path": None,
                }
            )
            continue

        accession = str(row.get("accessionNumber"))
        filing_date = str(row.get("filingDate"))
        form = safe_filename(str(row.get("form")))
        primary_desc = safe_filename(str(row.get("primaryDocDescription")))
        primary_doc = safe_filename(str(row.get("primaryDocument")))

        filename = safe_filename(
            f"{filing_date}_{form}_{primary_desc}_{accession}_{primary_doc}"
        )

        # primaryDocumentが .htm / .html 以外でも、HTML本文として保存したい場合は .html を付与
        if not filename.lower().endswith((".htm", ".html")):
            filename += ".html"

        path = out / filename

        if path.exists() and not overwrite:
            records.append(
                {
                    "accessionNumber": accession,
                    "filingDate": filing_date,
                    "form": row.get("form"),
                    "primaryDocDescription": row.get("primaryDocDescription"),
                    "primaryDocument": row.get("primaryDocument"),
                    "status": "exists",
                    "reason": None,
                    "source_url": str(url),
                    "saved_path": str(path),
                }
            )
            continue

        try:
            response = client.get(str(url))

            # 文字化け・encoding推定の副作用を避けるため、raw bytesで保存
            path.write_bytes(response.content)

            records.append(
                {
                    "accessionNumber": accession,
                    "filingDate": filing_date,
                    "form": row.get("form"),
                    "primaryDocDescription": row.get("primaryDocDescription"),
                    "primaryDocument": row.get("primaryDocument"),
                    "status": "downloaded",
                    "reason": None,
                    "source_url": str(url),
                    "saved_path": str(path),
                    "content_type": response.headers.get("Content-Type"),
                    "bytes": len(response.content),
                }
            )

        except Exception as e:
            records.append(
                {
                    "accessionNumber": accession,
                    "filingDate": filing_date,
                    "form": row.get("form"),
                    "primaryDocDescription": row.get("primaryDocDescription"),
                    "primaryDocument": row.get("primaryDocument"),
                    "status": "failed",
                    "reason": repr(e),
                    "source_url": str(url),
                    "saved_path": str(path),
                }
            )

    return pd.DataFrame(records)


def get_monthly_primary_doc_htmls(
    cik: str | int,
    year_month: str,
    primary_doc_description: str | list[str] | None = None,
    match_mode: MatchMode = "exact",
    form_types: list[str] | None = None,
    user_agent: str = USER_AGENT,
    out_dir: str | Path = "edgar_primary_doc_htmls",
    overwrite: bool = False,
) -> dict[str, Any]:
    """
    CIK × 月間 × primaryDocDescription でfilingを抽出し、
    selected_filingsの primary_doc_url のHTML本文のみ保存する。
    """
    client = EdgarClient(user_agent=user_agent, sleep_sec=SLEEP_SEC)

    cik10, entity_json, all_filings = get_all_submission_metadata(
        cik=cik,
        client=client,
    )

    selected_filings = filter_filings_by_month_and_primary_doc_description(
        filings=all_filings,
        year_month=year_month,
        primary_doc_description=primary_doc_description,
        match_mode=match_mode,
        form_types=form_types,
    )

    download_log = download_selected_primary_doc_htmls(
        selected_filings=selected_filings,
        client=client,
        out_dir=out_dir,
        overwrite=overwrite,
    )

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    selected_filings.to_csv(out / "selected_filings.csv", index=False)
    download_log.to_csv(out / "download_log.csv", index=False)

    return {
        "cik": cik10,
        "entity_name": entity_json.get("name"),
        "filings": selected_filings,
        "download_log": download_log,
        "out_dir": str(out),
    }

In [ ]:
get_monthly_primary_doc_htmls(
    cik="0001419828", # https://www.sec.gov/edgar/search/
    year_month="2026-06",
    primary_doc_description="424B2",
    match_mode="exact",
    user_agent=USER_AGENT,
    out_dir="/content/",
)

/tmp/ipykernel_15787/3238954183.py:143: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


  0%|          | 0/798 [00:00<?, ?it/s]

{'cik': '0001419828',
 'entity_name': 'GS Finance Corp.',
 'filings':           accessionNumber  filingDate reportDate        acceptanceDateTime  \
 0    0001193125-26-248926  2026-06-01             2026-05-29T21:37:11.000Z   
 1    0001193125-26-249004  2026-06-01             2026-05-29T22:32:59.000Z   
 2    0001193125-26-249126  2026-06-01             2026-05-29T23:21:54.000Z   
 3    0001193125-26-249476  2026-06-01             2026-06-01T10:00:28.000Z   
 4    0001193125-26-250289  2026-06-01             2026-06-01T14:51:14.000Z   
 ..                    ...         ...        ...                       ...   
 793  0001193125-26-276064  2026-06-18             2026-06-18T21:00:33.000Z   
 794  0001193125-26-276066  2026-06-18             2026-06-18T21:00:58.000Z   
 795  0001193125-26-276073  2026-06-18             2026-06-18T21:03:15.000Z   
 796  0001193125-26-276083  2026-06-18             2026-06-18T21:08:52.000Z   
 797  0001193125-26-276090  2026-06-18             2026-06-18T